# TerraSpectra – Day 3: Ground-Truth Annotation Analysis

**Purpose:** inspect the plant annotation coordinates, create a binary annotation mask, overlay annotations on the plant image, and extract spectral measurements for the annotated pixels.

## Important: connect the Day 2 normalized cube

A variable created in `TerraSpectra_Day02_Preprocessing.ipynb` does **not** automatically exist in this separate notebook. Day 3 therefore loads the normalized hyperspectral cube from a saved `.npy` file.

Before running this notebook, if you did not save the normalized cube in Day 2, return to Day 2 and run this after normalization:

```python
# Use the actual variable that contains your normalized 512x512x204 array.
np.save("/content/hyperspectral_normalized.npy", hyperspectral_data)
```

If your normalized array has a different variable name, replace `hyperspectral_data` with that name. Then download `hyperspectral_normalized.npy` from Colab and upload it when prompted below.

Run the cells **from top to bottom**. Do not skip the setup and data-loading cells.

In [ ]:
# 1. Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

print("TerraSpectra Day 3 - libraries imported successfully!")

In [ ]:
# 2. Set file paths and upload any missing input files
annotation_path = "/content/pos_Plant21.csv"
image_path = "/content/2047.png"
normalized_path = "/content/hyperspectral_normalized.npy"

required_files = [annotation_path, image_path, normalized_path]
missing_files = [path for path in required_files if not os.path.exists(path)]

if missing_files:
    print("Please upload the following missing file(s):")
    for path in missing_files:
        print("-", os.path.basename(path))
    uploaded = files.upload()

    # Uploaded files are saved in /content by Colab.
    print("\nUploaded files:", list(uploaded.keys()))

still_missing = [path for path in required_files if not os.path.exists(path)]
if still_missing:
    raise FileNotFoundError(
        "Missing required file(s): " +
        ", ".join(os.path.basename(path) for path in still_missing) +
        ". Check the filenames and upload them to this Colab session."
    )

print("\nAll three input files are available.")

## 1. Load and clean the ground-truth coordinates

The CSV has no header row. Its first row (`174, 217`) is a real coordinate, not a column heading. We must load it with `header=None` so that all **244** coordinate rows are retained.

In [ ]:
# 3. Load annotation CSV correctly
annotations = pd.read_csv(
    annotation_path,
    header=None,
    names=["row", "col"]
)

# Ensure coordinates are numeric and remove no rows silently.
annotations["row"] = pd.to_numeric(annotations["row"], errors="raise").astype(int)
annotations["col"] = pd.to_numeric(annotations["col"], errors="raise").astype(int)

print("Annotation shape:", annotations.shape)
print("\nFirst 10 coordinates:")
display(annotations.head(10))

print("\nData types:")
print(annotations.dtypes)

print("\nMissing values:")
print(annotations.isnull().sum())

print("\nCoordinate summary:")
display(annotations.describe())

In [ ]:
# 4. Check for duplicate and invalid coordinates
height, width = 512, 512

duplicate_count = annotations.duplicated(subset=["row", "col"]).sum()

invalid_coordinates = annotations[
    (annotations["row"] < 0) | (annotations["row"] >= height) |
    (annotations["col"] < 0) | (annotations["col"] >= width)
]

print("Total annotation rows:", len(annotations))
print("Unique coordinate pairs:", annotations[["row", "col"]].drop_duplicates().shape[0])
print("Duplicate coordinate rows:", duplicate_count)
print("Invalid coordinates:", len(invalid_coordinates))

assert len(annotations) == 244, "Expected 244 annotation rows; check the CSV file."
assert len(invalid_coordinates) == 0, "Some coordinates are outside the 512x512 image."

print("\nCoordinate validation successful.")

In [ ]:
# 5. Plot the annotation coordinate distribution
plt.figure(figsize=(9, 7))
plt.scatter(annotations["col"], annotations["row"], s=15, color="red")
plt.title("TerraSpectra - Plant Annotation Coordinates")
plt.xlabel("Column (X)")
plt.ylabel("Row (Y)")
plt.grid(True)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Visualize annotations on the plant image

The provided image is RGBA (`512 × 512 × 4`). Matplotlib can display this directly; do not stack extra channels onto it.

In [ ]:
# 6. Load plant image and overlay annotation coordinates
plant_image = np.array(Image.open(image_path))
print("Plant image shape:", plant_image.shape)

assert plant_image.shape[:2] == (512, 512), (
    "Plant image dimensions do not match the annotation coordinate system."
)

plt.figure(figsize=(10, 8))
plt.imshow(plant_image)
plt.scatter(
    annotations["col"],
    annotations["row"],
    s=12,
    color="red",
    label="Annotated plant pixels"
)
plt.title("TerraSpectra - Plant Annotation Overlay")
plt.legend()
plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Load and verify the Day 2 normalized hyperspectral cube

This is the fix for the `NameError: hyperspectral_data is not defined`. We explicitly load the saved `.npy` array into that variable in this notebook.

In [ ]:
# 7. Load normalized hyperspectral data saved from Day 2
hyperspectral_data = np.load(normalized_path)

print("Loaded hyperspectral shape:", hyperspectral_data.shape)
print("Data type:", hyperspectral_data.dtype)
print("Mean:", float(np.mean(hyperspectral_data)))
print("Standard deviation:", float(np.std(hyperspectral_data)))

assert hyperspectral_data.shape == (512, 512, 204), (
    "Expected normalized data shape (512, 512, 204). "
    "Please check that you uploaded the Day 2 normalized .npy file."
)

if not np.isfinite(hyperspectral_data).all():
    raise ValueError("The hyperspectral cube contains NaN or infinite values.")

print("\nNormalized hyperspectral dataset loaded and verified.")

## 4. Create and visualize the binary annotation mask

Mask convention:
- `1` = pixel included in the supplied annotation coordinates
- `0` = pixel not included in those coordinates

**Interpretation:** the supplied coordinates identify an annotated plant region. They do not, by themselves, prove that those pixels are diseased.

In [ ]:
# 8. Create binary annotation mask
annotation_mask = np.zeros(
    hyperspectral_data.shape[:2],
    dtype=np.uint8
)

rows = annotations["row"].to_numpy()
cols = annotations["col"].to_numpy()
annotation_mask[rows, cols] = 1

print("Annotation mask shape:", annotation_mask.shape)
print("Annotated pixels in mask:", int(annotation_mask.sum()))
print("Unannotated pixels:", int(annotation_mask.size - annotation_mask.sum()))

In [ ]:
# 9. Display binary annotation mask
plt.figure(figsize=(8, 7))
plt.imshow(annotation_mask, cmap="gray", vmin=0, vmax=1)
plt.title("TerraSpectra - Ground-Truth Annotation Mask")
plt.xlabel("Column")
plt.ylabel("Row")
plt.colorbar(label="Mask value (0 or 1)")
plt.tight_layout()
plt.show()

## 5. Extract spectral information from annotated pixels

Each annotated pixel has 204 spectral measurements. With 244 annotation rows, the extracted feature matrix should contain `244 × 204 = 49,776` spectral values (plus two coordinate columns in the table).

In [ ]:
# 10. Extract 204 spectral bands for every annotated coordinate
pixel_spectra = hyperspectral_data[rows, cols, :]

spectral_columns = [f"band_{i}" for i in range(1, hyperspectral_data.shape[2] + 1)]

spectral_df = pd.DataFrame(pixel_spectra, columns=spectral_columns)
spectral_df.insert(0, "col", cols)
spectral_df.insert(0, "row", rows)

print("Extracted spectral dataset shape:", spectral_df.shape)
print("Number of annotated pixels:", pixel_spectra.shape[0])
print("Number of spectral bands:", pixel_spectra.shape[1])
print("Total spectral measurements:", pixel_spectra.size)
display(spectral_df.head())

In [ ]:
# 11. Plot mean spectral signature across annotated pixels
mean_spectrum = pixel_spectra.mean(axis=0)
std_spectrum = pixel_spectra.std(axis=0)

band_numbers = np.arange(1, pixel_spectra.shape[1] + 1)

plt.figure(figsize=(12, 6))
plt.plot(band_numbers, mean_spectrum, label="Mean annotated-region spectrum")
plt.fill_between(
    band_numbers,
    mean_spectrum - std_spectrum,
    mean_spectrum + std_spectrum,
    alpha=0.2,
    label="±1 standard deviation"
)
plt.title("TerraSpectra - Mean Annotated-Region Spectral Signature")
plt.xlabel("Spectral Band")
plt.ylabel("Normalized Reflectance")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 12. Generate spectral statistics
spectral_summary = spectral_df[spectral_columns].describe().T
display(spectral_summary)

print("Spectral values contain missing data:",
      bool(np.isnan(pixel_spectra).any()))
print("Spectral values contain infinity:",
      bool(np.isinf(pixel_spectra).any()))

## 6. Save Day 3 outputs

The following files are created in `/content/TerraSpectra_Day03/`. Download the ZIP after the cell finishes and add the relevant outputs to your project repository.

In [ ]:
# 13. Save cleaned annotations, mask, spectra, and summary
output_dir = "/content/TerraSpectra_Day03"
os.makedirs(output_dir, exist_ok=True)

annotations.to_csv(
    os.path.join(output_dir, "plant_annotations.csv"),
    index=False
)

np.save(
    os.path.join(output_dir, "plant_annotation_mask.npy"),
    annotation_mask
)

spectral_df.to_csv(
    os.path.join(output_dir, "annotated_plant_spectra.csv"),
    index=False
)

spectral_summary.to_csv(
    os.path.join(output_dir, "spectral_summary.csv")
)

pd.DataFrame({
    "band": band_numbers,
    "mean": mean_spectrum,
    "std": std_spectrum
}).to_csv(
    os.path.join(output_dir, "mean_spectral_signature.csv"),
    index=False
)

print("Saved output files:")
for filename in sorted(os.listdir(output_dir)):
    print("-", filename)

In [ ]:
# 14. Create a ZIP archive for download
import shutil

zip_path = shutil.make_archive(
    "/content/TerraSpectra_Day03_outputs",
    "zip",
    output_dir
)

print("Output archive created:", zip_path)
files.download(zip_path)

In [ ]:
# 15. Final Day 3 verification
print("=" * 55)
print("TERRASPECTRA - DAY 3 COMPLETION REPORT")
print("=" * 55)
print("Hyperspectral image shape:", hyperspectral_data.shape)
print("Annotation table shape:", annotations.shape)
print("Annotation mask shape:", annotation_mask.shape)
print("Annotated pixels in mask:", int(annotation_mask.sum()))
print("Extracted spectral table shape:", spectral_df.shape)
print("Spectral bands:", len(spectral_columns))
print("Invalid coordinates:", len(invalid_coordinates))
print("Missing coordinate values:", int(annotations.isnull().sum().sum()))
print("\nDAY 3 ANALYSIS COMPLETED.")